# =============================== # Modelling Notebook # ===============================

## Objectives

- Fit and evaluate regression models to predict SalePrice.
- Apply feature engineering pipeline developed earlier.
- Compare candidate regression models using cross-validation.
- Select best model and optimise hyperparameters.
- Export final train/test sets, pipelines, and feature importance plot.


## Inputs 
- outputs/datasets/processed/TrainSet.csv
- outputs/datasets/processed/TestSet.csv

## Outputs
- Train set (features and target)
- Test set (features and target)
- Data cleaning and Feature Engineering pipeline
- Modeling pipeline
- Feature importance plot

### Import Cell

In [31]:
import sys
!"{sys.executable}" -m pip install xgboost


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split

# Pipelines
from sklearn.pipeline import Pipeline

# Feature Engineering
from feature_engine.selection import SmartCorrelatedSelection
from feature_engine.outliers import Winsorizer
from sklearn.preprocessing import PowerTransformer

# Feature Scaling
from sklearn.preprocessing import StandardScaler

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor

# Hyperparameter Search
from sklearn.model_selection import GridSearchCV

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

---

## Change working directory

We need to set the current working directory to the parent folder for consistency.

In [33]:
# Check current directory
current_dir = os.getcwd()
print("Current directory:", current_dir)

# Move to parent directory
os.chdir(os.path.dirname(current_dir))
print("New current directory:", os.getcwd())

Current directory: c:\Users\david
New current directory: c:\Users


Confirm the new current directory

In [34]:
current_dir = os.getcwd()
current_dir

'c:\\Users'

## Load Cleaned Data and Split into Trained and Test Sets



In [35]:
BASE_DIR = Path(r"C:\Users\david\Portfolio 5\heritage-housing")
data_path = BASE_DIR / "outputs/datasets/cleaned/CleanedData.csv"

# Load data
df = pd.read_csv(data_path)

In [36]:
# Split into Train and Test sets
TrainSet, TestSet = train_test_split(df, test_size=0.2, random_state=42)

print("TrainSet shape:", TrainSet.shape)
print("TestSet shape:", TestSet.shape)

target = "SalePrice"

X_train = TrainSet.drop(columns=[target])
y_train = TrainSet[target]

X_test = TestSet.drop(columns=[target])
y_test = TestSet[target]

TrainSet shape: (1168, 22)
TestSet shape: (292, 22)


---

## Helper Functions

These functions help check missing values and generate diagnostic plots for numeric and categorical features.

In [37]:
def evaluate_regression (x, y, pipeline):
    """
    Evaluates a trained regression model using common performance metrics.
    """
    y_pred = pipeline.predict(X)

    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)

    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)

    return mae, rmse, r2

def plot_predictions(x, y, pipeline):
    """
    Plots actual vs predicted values for regression model evaluation.
    """
    y_pred = pipeline.predict(X)

    plt.figure(figsize=(6,6))
    plt.scatter(y, y_pred)
    plt.xlabel("Actual Values")
    plt.ylabel("Predicted Values")
    plt.title("Actual vs Predicted")
    plt.show()

def plot_residuals(x, y, pipeline):
    """
    Plots residuals (errors) of a regression model to assess model fit.
    Residuals are calculated as: actual - predicted.
    """
    y_pred = pipeline.predict(X)
    residuals = y - y_pred

    plt.figure(figsize=(6,4))
    sns.histplot(residuals, kde= true)
    plt.xlabel("Residuals (Actual - Predicted)")
    plt.title("Residual Distribution")
    plt.show()

def save_model_outputs(model, file_path):
    """
    Saves a trained machine learning pipeline to a specified directory.
    """

    os.makedirs(file_path, exist_ok=True)

    joblib.dump(model, f"{file_path}/model.pkl")

    print(f"Model successfully saved to: {file_path}")

# ===============================
## Feature Engineering Pipeline
# ===============================

### Phase 1: Transformation Exploration & Selection

In [38]:
def PipelineDataCleaningAndFeatureEngineering():
    """
    Light feature engineering pipeline for regression.
    """

    pipeline = Pipeline([
        ("correlation_selection",
         SmartCorrelatedSelection(
             variables=None,
             method="spearman",
             threshold=0.6,
             selection_method="variance"
         )
        ),

        ("outliers",
         Winsorizer(
             capping_method="iqr",
             tail="both",
             fold=1.5,
             variables=['GrLivArea', 'LotArea', 'TotalBsmtSF']
         )
        )
    ])

    return pipeline

In [39]:
pipeline_fe = PipelineDataCleaningAndFeatureEngineering()

X_train = pipeline_fe.fit_transform(X_train)
X_test = pipeline_fe.transform(X_test)

### Model Benchmarking

#### Define Candidate Models

In [40]:
models_quick_search = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "RandomForestRegressor": RandomForestRegressor(random_state=42),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=42),
    "XGBRegressor": XGBRegressor(random_state=42)
}

### Define Hyperparameter Grids (Baseline)

In [41]:
params_quick_search = {
    "LinearRegression": {},
    "Ridge": {},
    "Lasso": {},
    "RandomForestRegressor": {},
    "GradientBoostingRegressor": {},
    "ExtraTreesRegressor": {},
    "XGBRegressor": {}
}

### Run GridSearchCV (Quick Benchmark)

#### Custom class for Hyperparameter Optimization

In [42]:
def PipelineReg(model):
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("feat_selection", SelectFromModel(model)),
        ("model", model),
    ])
    return pipeline


class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key}\n")

            model = PipelineReg(self.models[key])
            params = self.params[key]

            gs = GridSearchCV(
                estimator=model,
                param_grid=params,
                cv=cv,
                n_jobs=n_jobs,
                verbose=verbose,
                scoring=scoring
            )

            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):

        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []

            for i in range(self.grid_searches[k].cv):
                key = f"split{i}_test_score"
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)

            for p, s in zip(params, all_scores):
                rows.append(row(k, s, p))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)

        columns = ['estimator', 'min_score', 'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]

        return df[columns], self.grid_searches

In [43]:
search = HyperparameterOptimizationSearch(
    models=models_quick_search,
    params=params_quick_search
)

search.fit(
    X_train,
    y_train,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)


Running GridSearchCV for LinearRegression

Fitting 5 folds for each of 1 candidates, totalling 5 fits


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\david\Portfolio 5\heritage-housing\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\david\Portfolio 5\heritage-housing\venv\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\david\Portfolio 5\heritage-housing\venv\Lib\site-packages\sklearn\pipeline.py", line 613, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "

### Evaluate Model Performance

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

grid_search_summary

### Select Best Performing Model

In [ ]:
best_model = grid_search_summary.iloc[0, 0]
best_model

### Define Hyperparameter Grid for Best Model

In [ ]:
models_search = {
    "XGBRegressor": XGBRegressor(random_state=42)
}

params_search = {
    "XGBRegressor": {
        "model__learning_rate": [0.01, 0.1],
        "model__max_depth": [3, 5, 10]
    }
}

### Run GridSearchCV (Tuned Model)

In [ ]:
search = HyperparameterOptimizationSearch(
    models=models_search,
    params=params_search
)

search.fit(
    X_train,
    y_train,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)

### Evaluate Tuned Model Performance

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

grid_search_summary

### Extract Best Model Pipeline

In [ ]:
best_model = grid_search_summary.iloc[0, 0]

pipeline_clf = grid_search_pipelines[best_model].best_estimator_

pipeline_clf

## Model Evaluation 

### Evaluate Model on Train Set

In [ ]:
evaluate_regression(X_train, y_train, pipeline_clf)

### Evaluate Model on Test Set


In [ ]:
evaluate_regression(X_test, y_test, pipeline_clf)

---

### Plot Actual vs Predicted Values

In [ ]:
plot_predictions(X_test, y_test, pipeline_clf)

* Numerical transformations applied (Box-Cox + Yeo-Johnson)


### Plot Residual Distribution

In [ ]:
plot_residuals(X_test, y_test, pipeline_clf)

Correlated feature sets:
[{'GrLivArea', 'GarageArea', 'OverallQual', 'YearBuilt', 'SalePrice'}, {'YearRemodAdd', 'GarageYrBlt'}, {'1stFlrSF', 'TotalBsmtSF'}]
Features to drop:
['GarageArea', 'YearBuilt', 'OverallQual', 'GrLivArea', 'YearRemodAdd', 'TotalBsmtSF']


## Feature Importance

### Extract Feature Importance from the Model

In [ ]:
best_features = X_train.columns

df_feature_importance = (
    pd.DataFrame({
        "Feature": best_features,
        "Importance": pipeline_clf["model"].feature_importances_
    })
    .sort_values(by="Importance", ascending=False)
)

df_feature_importance

c:\Users\david\Portfolio 5\heritage-housing\venv\Lib\site-packages\feature_engine\encoding\base_encoder.py:260: UserWarning: During the encoding, NaN values were introduced in the feature(s) BsmtExposure, BsmtFinType1, GarageFinish.
  warnings.warn(
c:\Users\david\Portfolio 5\heritage-housing\venv\Lib\site-packages\feature_engine\encoding\base_encoder.py:260: UserWarning: During the encoding, NaN values were introduced in the feature(s) BsmtExposure, BsmtFinType1, GarageFinish.
  warnings.warn(


### Plot Feature Importance

---

In [ ]:
plt.figure(figsize=(12, 6))
df_feature_importance.plot(kind="bar", x="Feature", y="Importance", legend=False)
plt.title("Feature Importance")
plt.ylabel("Importance Score")
plt.tight_layout()
plt.show()

* Winsorization applied!
